# Task 2 - Generative AI: Domain-Specific Fine-Tuning Pipeline
CDAZZDEV Senior MLE Assessment

Use case: **SEC-style Risk-Factor Clause Classification & Explanation**
(see `problem_statement.md`). Teacher model: Groq `llama-3.3-70b-versatile`.
Student model (fine-tuned): `microsoft/Phi-3-mini-4k-instruct`.

**Set the runtime to a GPU before running 2B/2C:**
`Runtime > Change runtime type > T4 GPU`. 2A and the bonus RAG section run
fine on CPU. Cell outputs are kept (not cleared) as required.

## 0. Setup

In [1]:
!pip install -q groq pydantic pandas rouge-score chromadb \
    transformers peft trl bitsandbytes accelerate datasets

In [2]:
import os
# In Colab: click the key icon in the left sidebar -> add secret GROQ_API_KEY,
# then uncomment the two lines below. Also add HF_TOKEN as a secret if you
# want to push the merged model to Hugging Face Hub.
# from google.colab import userdata
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

assert os.environ.get("GROQ_API_KEY"), "Set GROQ_API_KEY before continuing."

## 1. Task 2A - Use Case Definition and Dataset Engineering

Full teacher system prompt used for synthetic data generation (also documented
in `CITATIONS.md` and `dataset_generation.py`):

```
See TEACHER_GENERATION_PROMPT in dataset_generation.py for the exact template (it is parameterised per sector/risk_category/style).
```

In [3]:
from dataset_generation import generate_dataset, diversity_report, split_and_save
import json

dataset = generate_dataset(min_examples=100)
print(f"Generated {len(dataset)} examples")
print(json.dumps(diversity_report(dataset), indent=2, default=str))

2026-10-06 13:39:24,674 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 13:39:26,539 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 13:39:28,377 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 13:39:30,708 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 13:39:33,070 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 13:39:34,765 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 13:39:36,388 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 13:39:38,006 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 13:39:39,951 [INFO] HTTP Request: POST https://api.groq.com/o

Generated 100 examples
{
  "n_examples": 100,
  "prompt_length_words": {
    "min": 52,
    "max": 124,
    "mean": 88.6
  },
  "top_keywords": [
    [
      "ability",
      63
    ],
    [
      "materially",
      51
    ],
    [
      "impair",
      46
    ],
    [
      "cash",
      45
    ],
    [
      "increase",
      35
    ],
    [
      "financial",
      35
    ],
    [
      "market",
      34
    ],
    [
      "operations",
      34
    ],
    [
      "any",
      33
    ],
    [
      "meet",
      33
    ],
    [
      "material",
      32
    ],
    [
      "regulatory",
      31
    ],
    [
      "prolonged",
      30
    ],
    [
      "capital",
      29
    ],
    [
      "adverse",
      29
    ]
  ],
  "risk_category_distribution": {
    "Reputational Risk": 13,
    "Credit Risk": 13,
    "Operational Risk": 9,
    "Cybersecurity Risk": 11,
    "Liquidity Risk": 14,
    "Market Risk": 17,
    "Regulatory/Legal Risk": 12,
    "Macroeconomic Risk": 11
  },
  "

In [4]:
sizes = split_and_save(dataset, out_dir="data")
print("Split sizes (80/10/10):", sizes)

2026-10-06 13:52:03,719 [INFO] Wrote 80 examples to data\train.jsonl
2026-10-06 13:52:03,726 [INFO] Wrote 10 examples to data\val.jsonl
2026-10-06 13:52:03,729 [INFO] Wrote 10 examples to data\test.jsonl


Split sizes (80/10/10): {'train': 80, 'val': 10, 'test': 10}


**Check:** `risk_category_distribution` should cover all 8 categories, and
`near_duplicate_pair_ratio` should be low. If it's high, the teacher model is
repeating itself -- consider raising temperature or adding more sectors/styles.

## 2. Task 2B - Fine-Tuning Execution (QLoRA, 4-bit NF4)

In [5]:
import torch
assert torch.cuda.is_available(), "No GPU detected -- set Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

AssertionError: No GPU detected -- set Runtime > Change runtime type > T4 GPU

In [ ]:
from fine_tune import run_fine_tuning

trainer, loss_history = run_fine_tuning()

**Check:** scroll the printed per-epoch loss above -- `eval_loss` must
decrease across the 3 epochs. The merged model is saved to
`phi3-risk-classifier-merged/` and pushed to Hugging Face Hub automatically
if `HF_TOKEN` is set.

## 3. Task 2C - Evaluation and Baseline Comparison

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import json

tokenizer = AutoTokenizer.from_pretrained("phi3-risk-classifier-merged")
ft_model = AutoModelForCausalLM.from_pretrained("phi3-risk-classifier-merged", device_map="auto", torch_dtype=torch.bfloat16)

base_tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
base_model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-4k-instruct", device_map="auto", torch_dtype=torch.bfloat16)

from dataset_generation import SYSTEM_CLASSIFICATION_PROMPT

def generate(model, tok, clause, max_new_tokens=150):
    messages = [{"role": "system", "content": SYSTEM_CLASSIFICATION_PROMPT}, {"role": "user", "content": clause}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

with open("data/test.jsonl", encoding="utf-8") as f:
    test_examples = [json.loads(l) for l in f]

clauses = [next(m["content"] for m in ex["messages"] if m["role"] == "user") for ex in test_examples]
references = [next(m["content"] for m in ex["messages"] if m["role"] == "assistant") for ex in test_examples]
gold_labels = [json.loads(r) for r in references]

base_preds = [generate(base_model, base_tokenizer, c) for c in clauses]
ft_preds = [generate(ft_model, tokenizer, c) for c in clauses]
print("Sample fine-tuned prediction:", ft_preds[0])

In [ ]:
from evaluate import rouge_comparison_table, judge_all, build_manual_review_sheet, compute_hallucination_rate

rouge_table = rouge_comparison_table(base_preds, ft_preds, references)
rouge_table

In [ ]:
judged = judge_all(clauses, gold_labels, ft_preds)
print("LLM-as-judge mean overall_score:", judged["overall_score"].mean())
print("category_correct rate:", judged["category_correct"].mean())
print("grounded_in_clause rate:", judged["grounded_in_clause"].mean())
judged

### Manual hallucination-rate review

Fill in the `label` column below by hand for each row (`correct`,
`partially_correct`, or `hallucinated`) by actually reading the clause and
prediction -- this is intentionally not automated.

In [ ]:
review_sheet = build_manual_review_sheet(clauses, ft_preds, min_n=10)
review_sheet

In [ ]:
# After filling in review_sheet["label"] for every row above:
# review_sheet["label"] = ["correct", "correct", "hallucinated", ...]
# rate = compute_hallucination_rate(review_sheet)
# print(f"Hallucination rate: {rate}%")

### Qualitative analysis

Fill this in with real examples from the outputs above (see
`QUALITATIVE_ANALYSIS_TEMPLATE` in `evaluate.py` for the structure: one
paragraph on where fine-tuning helped with specific examples, one on
remaining failure modes and what would address them).

## 4. Bonus - RAG Fallback Layer

In [ ]:
from rag_fallback import build_vector_store, classify_with_rag_fallback

def ft_generate(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(ft_model.device)
    out = ft_model.generate(**inputs, max_new_tokens=100, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

collection = build_vector_store("data/train.jsonl")

# Pick a deliberately ambiguous/edge-case clause to demonstrate the fallback triggering
example_clause = clauses[0]
result = classify_with_rag_fallback(ft_generate, collection, example_clause, SYSTEM_CLASSIFICATION_PROMPT)
import json
print(json.dumps(result, indent=2, default=str))

**Check:** if `rag_triggered` is `True` above, compare `initial_prediction`
vs `rag_prediction` -- that's your required concrete before/after example.